# Where should I look for stellar oscillations?

This tutorial shows the basic ways to use AsteroScale. In this notebook we use it to predict some typical quantities about the oscillations in a solar-like oscillator, these include the approximate frequency of the oscillations, the band-width they appear ober and their amplitude as it would appear in a time series. Note oscillations are visible in both radial velocity and photometric time series, however in AsteroScale we currently only consider photometric variability. 

In [ ]:
import asteroscale as ast

## A quick point prediction

Plain numbers are treated as exact. Only quantities required by the requested outputs are needed, so parallax and extinction are not required for this seismic prediction.

In [ ]:
star = {"M": 1.1, "R": 1.3, "Teff": 6000, "FeH": 0.1}

prediction = ast.solve(
    star,
    want=["numax", "dnu", "FWHM_env", "A_env",
          "A_gran", "b_gran_low", "b_gran_high"],
    bandpass="TESS",
)
prediction

Oscillation power should peak near `numax`. A useful first search interval is approximately `numax ± FWHM_env/2`, although this is not a hard boundary. `A_env` is the maximum radial-mode RMS amplitude, not the total light-curve RMS.

In [ ]:
lower = prediction["numax"] - prediction["FWHM_env"] / 2
upper = prediction["numax"] + prediction["FWHM_env"] / 2
print(f"Suggested search interval: {lower:.0f}–{upper:.0f} microhertz")

## TESS and Kepler amplitudes

Kepler amplitudes are slightly larger because its response is bluer. The empirical Ball et al. relation and its hot-star suppression factor are otherwise unchanged.

In [ ]:
for bandpass in ("TESS", "Kepler"):
    amplitude = ast.solve(star, want=["A_env"], bandpass=bandpass)["A_env"]
    print(f"{bandpass:6s}: {amplitude:.2f} ppm")

## Propagating stellar-property uncertainties

A `(value, uncertainty)` pair represents a Gaussian measurement. AsteroScale returns samples rather than a single value.

In [ ]:
given = {
    "M": (1.10, 0.08),
    "R": (1.30, 0.04),
    "Teff": (6000, 80),
    "FeH": (0.10, 0.05),
}
samples = ast.solve(
    given, want=["numax", "dnu", "FWHM_env", "A_env", "A_gran"],
    preset="fast", seed=42,
)
ast.summarize(samples)

A predicted signal is not necessarily detectable. Compare the frequency range with the Nyquist frequency and resolution of the observations, and account for instrumental noise, cadence attenuation, dilution, activity, gaps and the observing window.